# Molecular embeddings — post-training analysis

Reads **only** saved `predictions_targets.pt` files containing `predictions`, `targets`, and `mask`. No original dataset, training logs, or model weights are loaded.

Five figures: (1) per-fold and mean masked MSE curves, (2) model MSE comparison across checkpoints, (3) Pearson(predictions, targets) across checkpoints, (4) final observed-vs-predicted scatter, (5) final fold-by-model heatmaps. Pearson is ordinary response correlation, **not** PanThera Bliss synergy R.

Numbered checkpoint folders are treated as epoch numbers. `final` is assigned epoch 50 to match the existing analysis convention; change `FINAL_EPOCH` if its actual epoch is known.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (10, 5), 'axes.grid': True, 'grid.alpha': 0.25})
BASE_PATH = Path('full-training-runs/main_run')
PREDICTIONS_ROOT = BASE_PATH / 'outputs' / 'predictions'
ANALYSIS_DIR = BASE_PATH / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
MODELS = ['3d_infomax', 'graph', 'trimnet']
FOLDS = range(1, 10)
FINAL_EPOCH = 50
if not PREDICTIONS_ROOT.exists():
    raise FileNotFoundError(f'Not found: {PREDICTIONS_ROOT.resolve()}')
print('Reading:', PREDICTIONS_ROOT.resolve())
print('Writing:', ANALYSIS_DIR.resolve())

## 1. Load masked predictions and calculate metrics
All calculations exclude padded entries (`mask=False`) and non-finite values.

In [ ]:
def load_payload(path):
    try: payload = torch.load(path, map_location='cpu', weights_only=True)
    except TypeError: payload = torch.load(path, map_location='cpu')
    if not isinstance(payload, dict): raise TypeError(f'{path}: expected dict')
    missing = {'predictions','targets','mask'} - set(payload)
    if missing: raise KeyError(f'{path}: missing {missing}')
    def arr(x): return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)
    p = arr(payload['predictions']).astype(float, copy=False)
    y = arr(payload['targets']).astype(float, copy=False)
    m = arr(payload['mask']).astype(bool, copy=False)
    if p.shape != y.shape: raise ValueError(f'{path}: prediction/target shape mismatch')
    m = np.squeeze(m)
    if m.shape != p.shape: m = np.broadcast_to(m, p.shape)
    valid = m & np.isfinite(p) & np.isfinite(y)
    return p, y, valid

def pearson(p, y):
    p, y = np.asarray(p).ravel(), np.asarray(y).ravel()
    good = np.isfinite(p) & np.isfinite(y); p, y = p[good], y[good]
    if len(p) < 2 or np.std(p) == 0 or np.std(y) == 0: return np.nan
    return float(np.corrcoef(p, y)[0, 1])

def metrics(p, y, mask):
    p, y = p[mask], y[mask]
    if not len(p): return {'n':0, 'mse':np.nan, 'mae':np.nan, 'pearson':np.nan}
    e = p-y
    return {'n':len(p), 'mse':float(np.mean(e**2)), 'mae':float(np.mean(np.abs(e))), 'pearson':pearson(p,y)}

## 2. Discover prediction files
Supports `<model>/main_run/fold_<n>/<checkpoint>/predictions_targets.pt` and nested intermediate run directories.

In [ ]:
records = []
for model in MODELS:
    root = PREDICTIONS_ROOT / model
    if not root.exists(): warnings.warn(f'Missing model folder: {root}'); continue
    for path in root.rglob('predictions_targets.pt'):
        fold_dir = next((x for x in path.parents if x.name.startswith('fold_')), None)
        if fold_dir is None: continue
        try: fold = int(fold_dir.name.split('_',1)[1])
        except ValueError: continue
        if fold not in FOLDS: continue
        name = path.parent.name
        if name.lower() == 'final': epoch, kind = FINAL_EPOCH, 'final'
        elif name.isdigit(): epoch, kind = int(name), 'epoch'
        else: continue
        records.append({'model':model,'fold':fold,'checkpoint':name,'epoch':epoch,'kind':kind,'path':path})
files = pd.DataFrame(records)
if files.empty: raise FileNotFoundError(f'No prediction files under {PREDICTIONS_ROOT}')
display(files[['model','fold','checkpoint','epoch','path']])

rows, final_data = [], {}
for r in files.itertuples(index=False):
    p,y,m = load_payload(r.path)
    rows.append({'model':r.model,'fold':r.fold,'checkpoint':r.checkpoint,'kind':r.kind,'epoch':r.epoch,**metrics(p,y,m)})
    if r.kind == 'final': final_data[(r.model,r.fold)] = (p[m].ravel(),y[m].ravel())
metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(ANALYSIS_DIR / 'checkpoint_metrics.csv', index=False)
curve_df = metrics_df.sort_values(['model','fold','epoch','kind']).drop_duplicates(['model','fold','epoch'],keep='last')
display(metrics_df.head())

## 3. Figure 1 — MSE by fold and fold average
Individual fold curves are shown lightly; the bold curve is the unweighted fold mean with ±1 SD.

In [ ]:
fig, axes = plt.subplots(1,len(MODELS),figsize=(6*len(MODELS),5),sharey=True)
for ax, model in zip(axes,MODELS):
    d = curve_df[curve_df.model==model]
    for fold, fd in d.groupby('fold'):
        fd=fd.sort_values('epoch'); ax.plot(fd.epoch,fd.mse,marker='o',alpha=.4,lw=1,label=f'Fold {fold}')
    a=d.groupby('epoch').mse.agg(['mean','std']).reset_index(); a['std']=a['std'].fillna(0)
    ax.plot(a.epoch,a['mean'],color='black',marker='o',lw=2.6,label='Fold mean')
    ax.fill_between(a.epoch,a['mean']-a['std'],a['mean']+a['std'],color='black',alpha=.12)
    ax.set(title=model,xlabel='Checkpoint / epoch',ylabel='Masked MSE'); ax.legend(fontsize=7,ncol=2)
fig.tight_layout(); fig.savefig(ANALYSIS_DIR/'01_mse_by_fold.png',dpi=180,bbox_inches='tight'); plt.show()

## 4. Figure 2 — Model comparison across checkpoints
Mean fold MSE with ±1 SD. Lower is better.

In [ ]:
fig,ax=plt.subplots(figsize=(9,5.5))
for model in MODELS:
    d=curve_df[curve_df.model==model]; a=d.groupby('epoch').mse.agg(['mean','std']).reset_index(); a['std']=a['std'].fillna(0)
    ax.plot(a.epoch,a['mean'],marker='o',lw=2,label=model); ax.fill_between(a.epoch,a['mean']-a['std'],a['mean']+a['std'],alpha=.13)
ax.set(title='Model comparison — masked MSE',xlabel='Checkpoint / epoch',ylabel='Mean fold MSE'); ax.legend()
fig.tight_layout(); fig.savefig(ANALYSIS_DIR/'02_model_comparison_mse.png',dpi=180,bbox_inches='tight'); plt.show()

## 5. Figure 3 — Pearson correlation across checkpoints
Pearson is calculated directly between each checkpoint's predictions and targets. Higher is stronger linear association; it does not by itself imply lower error.

In [ ]:
fig,ax=plt.subplots(figsize=(9,5.5))
for model in MODELS:
    d=curve_df[curve_df.model==model]; a=d.groupby('epoch').pearson.agg(['mean','std']).reset_index(); a['std']=a['std'].fillna(0)
    ax.plot(a.epoch,a['mean'],marker='o',lw=2,label=model); ax.fill_between(a.epoch,a['mean']-a['std'],a['mean']+a['std'],alpha=.13)
ax.set(title='Pearson correlation: predictions vs targets',xlabel='Checkpoint / epoch',ylabel='Mean fold Pearson r',ylim=(-1.05,1.05)); ax.legend()
fig.tight_layout(); fig.savefig(ANALYSIS_DIR/'03_pearson_by_checkpoint.png',dpi=180,bbox_inches='tight'); plt.show()

## 6. Figure 4 — Final observed vs predicted
Scatter is deterministically subsampled for display when necessary; Pearson r in each title is calculated on all valid final points.

In [ ]:
def final_points(model):
    parts=[final_data[(model,f)] for f in FOLDS if (model,f) in final_data]
    if not parts: return np.array([]),np.array([])
    return np.concatenate([x[0] for x in parts]),np.concatenate([x[1] for x in parts])
fig,axes=plt.subplots(1,len(MODELS),figsize=(6*len(MODELS),5))
for ax,model in zip(axes,MODELS):
    p,y=final_points(model)
    if not len(p): ax.set_title(f'{model}: no final data'); continue
    r=pearson(p,y); rng=np.random.default_rng(42)
    if len(p)>100000: idx=rng.choice(len(p),100000,replace=False); pp,yy=p[idx],y[idx]
    else: pp,yy=p,y
    lo=min(pp.min(),yy.min()); hi=max(pp.max(),yy.max())
    ax.scatter(yy,pp,s=3,alpha=.12,rasterized=True); ax.plot([lo,hi],[lo,hi],'--',color='black')
    ax.set(title=f'{model} — Pearson r={r:.4f}',xlabel='Target',ylabel='Prediction')
fig.tight_layout(); fig.savefig(ANALYSIS_DIR/'04_final_scatter.png',dpi=180,bbox_inches='tight'); plt.show()

## 7. Figure 5 — Final fold-by-model heatmaps
Highlights fold-level differences in error and prediction-target correlation.

In [ ]:
final_df=metrics_df[metrics_df.kind=='final'].copy()
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,metric,title,cmap in [(axes[0],'mse','Final masked MSE','viridis'),(axes[1],'pearson','Final Pearson r','coolwarm')]:
    mat=final_df.pivot(index='fold',columns='model',values=metric).reindex(columns=MODELS).sort_index()
    im=ax.imshow(mat.values,aspect='auto',interpolation='nearest',cmap=cmap)
    ax.set_xticks(range(len(mat.columns)),mat.columns,rotation=25); ax.set_yticks(range(len(mat.index)),[f'Fold {x}' for x in mat.index]); ax.set_title(title)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v=mat.iloc[i,j]
            if pd.notna(v): ax.text(j,i,f'{v:.3f}',ha='center',va='center',fontsize=8)
    fig.colorbar(im,ax=ax,shrink=.8)
fig.tight_layout(); fig.savefig(ANALYSIS_DIR/'05_final_fold_heatmaps.png',dpi=180,bbox_inches='tight'); plt.show()

## 8. Summary and exports
Reports mean and SD across folds, plus pooled Pearson correlation across all final valid predictions.

In [ ]:
summary=(final_df.groupby('model').agg(folds=('fold','nunique'),mean_mse=('mse','mean'),sd_mse=('mse','std'),mean_mae=('mae','mean'),sd_mae=('mae','std'),mean_fold_pearson=('pearson','mean'),sd_fold_pearson=('pearson','std'),n_valid=('n','sum')).reset_index())
pooled=[]
for model in MODELS:
    p,y=final_points(model); pooled.append({'model':model,'pooled_pearson':pearson(p,y) if len(p) else np.nan,'pooled_n':len(p)})
summary=summary.merge(pd.DataFrame(pooled),on='model',how='left')
display(summary.round(6)); summary.to_csv(ANALYSIS_DIR/'final_model_summary.csv',index=False)
curve_df.to_csv(ANALYSIS_DIR/'checkpoint_metrics.csv',index=False)
print('Saved analysis to:',ANALYSIS_DIR.resolve())